# Numpy 之四：数组的运算

---

这一篇是 Numpy 的**核心**。前面讲过 list 和 ndarray 的运算语义完全不同，
现在把它彻底讲清楚。

## 学习目标

1. 分清 **元素级运算** 和 **聚合运算**
2. 彻底搞懂 **广播（broadcasting）** —— 这是最难也最有用的机制
3. 分清 **逐元素相乘 `*`** 和 **矩阵乘法 `@`**
4. 知道 `axis` 到底在"沿着哪个方向"算


In [ ]:
import numpy as np

arr1 = np.array([1, 2, 3, 4, 5])
arr2 = np.array([6, 7, 8, 9, 10])
print('arr1 =', arr1)
print('arr2 =', arr2)


---
## 一、元素级运算（element-wise）

**规则：形状相同的两个数组，对应位置一对一运算。**

这是 Numpy 和 list 最大的区别。回头看：

```python
[1,2,3] + [4,5,6]      # list：拼接 -> [1,2,3,4,5,6]
np.array([1,2,3]) + np.array([4,5,6])   # ndarray：逐项加 -> [5,7,9]
```


In [ ]:
print('arr1 + arr2 =', arr1 + arr2, '  <- 对应元素相加')
print('arr1 - arr2 =', arr1 - arr2)
print('arr1 * arr2 =', arr1 * arr2, '  <- 逐项相乘，不是矩阵乘法！')
print('arr1 / arr2 =', arr1 / arr2, '  <- 结果自动变成浮点')


### 数组和"标量"运算：广播的最简单形式

一个数组和一个单独的数运算时，那个数会被"摊开"到每个位置：


In [ ]:
print('arr1 + 2 =', arr1 + 2, '  <- 每个元素都加 2')
print('arr1 * 3 =', arr1 * 3, '  <- 每个元素都乘 3')
print('arr1 ** 2 =', arr1 ** 2, '  <- 每个元素平方')
print('arr1 > 3 =', arr1 > 3, '  <- 每个元素比较，返回【布尔数组】')


**注意 `arr1 > 3` 返回的是一整个布尔数组**，不是单个 `True/False`。
这个布尔数组后面会变成"筛选器"，非常好用（见索引与切片那篇）。


In [ ]:
print('\n用这个布尔数组筛选：', arr1[arr1 > 3])


---
## 二、聚合运算（reduction）

**把一整个数组"压缩"成一个数**，这叫聚合 / 归约。

有两种等价写法：


In [ ]:
# 写法 A：方法（点号调用）
print('arr1.sum()  =', arr1.sum())
# 写法 B：函数（numpy 级调用）
print('np.sum(arr1)=', np.sum(arr1))


两种结果一模一样。**为什么有两个？**
- `np.sum(arr)` 是函数风格，**对所有"像数组的东西"都能用**（比如 list）。
- `arr.sum()` 是方法风格，面向对象一点，稍微快一点点。

常用聚合一览：


In [ ]:
print('sum  求和     :', arr1.sum())
print('mean 平均值   :', arr1.mean())
print('max  最大值   :', arr1.max())
print('min  最小值   :', arr1.min())
print('std  标准差   :', arr1.std())
print('var  方差     :', arr1.var())
print('prod 求积     :', arr1.prod())
print('argmax 最大值的下标 :', arr1.argmax())


### 方差到底是"除以 n"还是"除以 n-1"？

这是统计学里最常见的困惑，**必须说清楚**：

| 名称 | 公式 | 用途 | Numpy 里 |
|---|---|---|---|
| 总体方差 | 除以 **n** | 数据就是"全部"时 | `np.var(a, ddof=0)` ← **默认** |
| 样本方差 | 除以 **n-1** | 用样本推断总体时 | `np.var(a, ddof=1)` |

Numpy **默认 `ddof=0`（除以 n）**，而 pandas 默认 `ddof=1`。
这俩算出来不一样，跨库对比时要当心。


In [ ]:
a = np.array([1, 2, 3, 4, 5])
print('总体方差 ddof=0 :', np.var(a))
print('样本方差 ddof=1 :', np.var(a, ddof=1))
print('手算验证 ddof=0 ：', ((a - a.mean()) ** 2).sum() / 5)
print('手算验证 ddof=1 ：', ((a - a.mean()) ** 2).sum() / (5 - 1))


---
## 三、`axis`：沿着哪个方向"压扁"

二维数组有两个方向，`axis` 就是问："你要**消掉哪个维度**？"

口诀：**`axis` 指的是被消掉的那个轴**。

```
数组形状 (3, 4) ：
  axis=0 -> 消掉第 0 轴（行）-> 结果 4 个数，每个是"一列"的聚合
  axis=1 -> 消掉第 1 轴（列）-> 结果 3 个数，每个是"一行"的聚合
```

**记法：`axis=0` 就是"往下压"（跨行，得到每列的结果）；
`axis=1` 就是"往右压"（跨列，得到每行的结果）。**


In [ ]:
m = np.array([[1, 2, 3],
              [4, 5, 6]])
print('矩阵：')
print(m)
print('形状：', m.shape)

print('\naxis=1（消掉列，得到【每行】的和）：', m.sum(axis=1))
print('axis=0（消掉行，得到【每列】的和）：', m.sum(axis=0))

print('\n对照：每行的平均值 =', m.mean(axis=1))
print('对照：每列的平均值 =', m.mean(axis=0))

print('\n不写 axis（全部压成一个数）：', m.sum())


> **易错点**：`axis=0` 得到的结果长度是"列数"，`axis=1` 得到的长度是"行数"。
> 很多人第一次都会搞反，**画个 `(3,4)` 的格子按一遍**就记住了。


---
## 四、广播（broadcasting）—— 本篇的重点

**广播 = 形状不同也能算**，Numpy 会自动把"小的"撑大到"大的"。

### 规则（从右往左逐位比较）

两个数组的形状，**从最右边一位开始**逐位比较，
每一位要么"相等"，要么"其中一个是 1"，就**兼容**；否则报错。

### 例子 1：`(3,)` 和 `(3, 1)`

```
   (3, )        补成      (1, 3)
   (3, 1)                 (3, 1)
   比较（从右往左）：
     第1位：3 vs 1   -> 有 1，可以广播
     第2位：无 vs 3  -> 缺的当成 1，可以广播
   结果形状：(3, 3)
```


In [ ]:
arr5 = np.array([1, 2, 3])          # 形状 (3,)
arr6 = np.array([[4],
                 [5],
                 [6]])              # 形状 (3, 1)

print('arr5.shape =', arr5.shape)
print('arr6.shape =', arr6.shape)
print('\narr5 + arr6 =')
print(arr5 + arr6)
print('  <- 形状 (3,3)：arr5 被"复制"成 3 行，arr6 被"复制"成 3 列')


逐项验证一下第一行：`arr5 = [1,2,3]`，`arr6` 第一行是 `[4]`，
所以 `1+4, 2+4, 3+4 = 5, 6, 7`。第二行 `[5]`：`6, 7, 8`。第三行 `[6]`：`7, 8, 9`。

**这不就是"外积表"吗？** 对，广播干的就是这种事。


### 例子 2：给每行减去自己的均值（实战中最常见的用法）


In [ ]:
data = np.array([[10, 20, 30],
                 [2, 4, 6],
                 [100, 200, 300]])
print('原始数据：')
print(data)

row_means = data.mean(axis=1)                 # 形状 (3,)
print('\n每行的均值：', row_means, ' 形状', row_means.shape)

# row_means 形状是 (3,)，要减到 (3,3) 的每一行上，
# 必须先变成"列向量" (3,1) 才行
centered = data - row_means.reshape(3, 1)
print('\n每行减去自己的均值：')
print(centered)
print('\n验证：现在每行的均值都是 0 ->', centered.mean(axis=1))


**为什么不能直接 `data - row_means`？**
因为 `data` 是 `(3,3)`，`row_means` 是 `(3,)`。
从右往左比：`3 vs 3` 兼容，然后缺的一位当 1 -> 会变成"给**每列**减"，
不是我们想要的"给每行减"。所以必须 `reshape(3,1)` 把它竖起来。

**这个坑在数据标准化里天天遇到，记住 `reshape(-1, 1)` 这个写法最省事：**


In [ ]:
centered2 = data - row_means.reshape(-1, 1)   # -1 表示"这一维你自己算"
print('用 reshape(-1, 1) 效果一样：')
print(centered2)


### 例子 3：不兼容的形状会直接报错


In [ ]:
try:
    bad = np.array([1, 2, 3]) + np.array([1, 2])
    print(bad)
except ValueError as e:
    print('报错了（这是好事，说明形状真的不兼容）：')
    print(e)


`(3,)` 和 `(2,)` 从右往左第一位就是 `3 vs 2`，
既不相等也没有 1，**没法广播**，直接抛 `ValueError`。

> **好消息**：报错比"算出一个错的结果"安全得多。宁可报错，不要静默错。


---
## 五、矩阵运算：`*` 和 `@` 不是一回事！

这是初学者最容易搞混的地方，**必须刻进脑子**：

| 运算 | 符号 | 含义 |
|---|---|---|
| 逐元素相乘 | `a * b` | 对应位置相乘（形状要能广播） |
| 矩阵乘法 | `a @ b` 或 `np.dot(a, b)` | 线性代数意义的矩阵乘 |

**`*` 是"逐元素"，`@` 才是"矩阵乘法"。**


In [ ]:
arr7 = np.array([[1, 2, 3],
                 [4, 5, 6]])          # 形状 (2, 3)
arr8 = np.array([[7, 8],
                 [9, 10],
                 [11, 12]])           # 形状 (3, 2)

print('arr7 形状：', arr7.shape, '（2行3列）')
print('arr8 形状：', arr8.shape, '（3行2列）')

print('\n矩阵乘法 arr7 @ arr8 =')
print(arr7 @ arr8)
print('  <- 形状 (2,2)。规则：左边行 × 右边列，对应项相乘再求和')

print('\n和 np.dot 等价吗？', np.array_equal(arr7 @ arr8, np.dot(arr7, arr8)))


### 手算验证第一个元素

`arr7` 第一行 `[1, 2, 3]`，`arr8` 第一列 `[7, 9, 11]`：

```
1×7 + 2×9 + 3×11 = 7 + 18 + 33 = 58
```
对上了。

### 矩阵乘法的形状规则

```
(2, 3) @ (3, 2)  ->  (2, 2)
    ↑内维必须相等↑   ↑取外维的 2 和 2
```

**"内维相等，结果取外维"** —— 内维不相等就报错。


In [ ]:
try:
    print(arr7 @ arr7)      # (2,3) @ (2,3)：内维 3 vs 2，不相等
except ValueError as e:
    print('(2,3) @ (2,3) 报错：')
    print(e)


### 转置 `.T`

转置就是"行列互换"：第 i 行第 j 列 -> 第 j 行第 i 列。形状 `(2,3)` 变 `(3,2)`。


In [ ]:
print('arr7 形状：', arr7.shape)
print('arr7.T 形状：', arr7.T.shape)
print('\narr7：')
print(arr7)
print('\narr7.T：')
print(arr7.T)


一维数组的 `.T` 是"摆设"（还是它自己），因为一维没有行列之分：


In [ ]:
print('一维数组 .T 之后形状不变：', np.array([1, 2, 3]).T.shape)


---
## 六、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| `a * b` 当矩阵乘法用 | 结果形状不对 | 矩阵乘法用 `a @ b` |
| `(3,)` 当成列向量 | 广播方向反了 | 用 `a.reshape(-1, 1)` 竖起来 |
| `axis=0` / `axis=1` 记反 | 结果长度对不上 | `axis=0` 消行得列结果；`axis=1` 消列得行结果 |
| 跨库比方差 | numpy 和 pandas 结果不同 | numpy 默认 `ddof=0`，pandas 默认 `ddof=1` |
| 形状不兼容硬算 | `ValueError` | 先 `print(a.shape, b.shape)` 对一下 |
| `a / b` 全是整数以为得整数 | 实际得浮点 | 除法结果一定是浮点，想要整除用 `//` |
| 浮点比较用 `==` | 意外的 `False` | 用 `np.allclose` |
| `arr1 > 3` 当成单个布尔值 | 其实是数组 | 要"是否全部/存在"用 `np.all` / `np.any` |

---

## 七、动手练习

**练习 1**：有数组 `a = [1,2,3,4]`、`b = [10,20,30,40]`，
分别算出：逐项相乘、矩阵风格的"点积"（提示：一维用 `np.dot` 得一个数）。

**练习 2**：生成 `(4, 3)` 的随机数组，分别算出每行的和与每列的和，说清用了哪个 `axis`。

**练习 3**：`a = np.arange(6).reshape(2, 3)`，`b = np.array([10, 20, 30])`，
先预测 `a + b` 的形状，再运行验证。（提示：这是"给每列加一个数"）

**练习 4**：把 `(2, 3)` 和 `(3, 2)` 两个矩阵相乘，手算第一个元素，
用代码验证你是否算对。

**练习 5**：构造一个 `(5, 3)` 的矩阵，把每一行都减去该行的均值，
验证结果的每行均值都是 0。

**练习 6**：判断 `np.array([1,2,3]) + np.array([[1],[2],[3]])` 能不能广播，
结果形状是多少，先写答案再运行。

---

**答案思路（先自己做）**

```python
# 1
a = np.array([1,2,3,4]); b = np.array([10,20,30,40])
print(a * b)          # 逐项: [10 40 90 160]
print(np.dot(a, b))   # 点积: 300

# 3
a = np.arange(6).reshape(2, 3)
b = np.array([10, 20, 30])
print((a + b).shape)          # (2, 3)，给每一列加对应数

# 5
m = np.random.rand(5, 3)
print(np.allclose((m - m.mean(axis=1, keepdims=True)).mean(axis=1), 0))

# 6
# (3,) 配 (3,1) -> 广播成 (3,3)
```
